# KineticHawkes — Research Notebook

Unit tests and visualization for:
1. **Hawkes Process** — decay/fire equations, stationarity, intensity plots
2. **Order Book** — FIFO L3 reconstruction, queue tracking
3. **Kinetic Queue** — fill-probability model, power law
4. **Databento Data** — live inspection of MSFT MBO records
5. **Integration** — Hawkes intensity driven by real market events

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import databento as db
from dataclasses import dataclass, field
from collections import OrderedDict
from typing import Optional
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
matplotlib.use('Agg')   # non-interactive backend for clean nbconvert execution
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

NOTEBOOK_DIR = Path('/Users/paff/Desktop/Projects/kinetichawkes')
DBN_PATH = str(NOTEBOOK_DIR / 'data' / 'MSFT_20240303_20240314_mbo.dbn')

NANOS_PER_DOLLAR = 1_000_000_000
TICK_SIZE        = 100_000_000   # $0.10

SESSION_OPEN_NS  = int(13.5 * 3600 * 1e9)   # 13:30:00 UTC
SESSION_CLOSE_NS = int(21.0 * 3600 * 1e9)   # 21:00:00 UTC
NANOS_PER_DAY    = int(24  * 3600 * 1e9)

def in_session(ts_ns: int) -> bool:
    tod = ts_ns % NANOS_PER_DAY
    return SESSION_OPEN_NS <= tod < SESSION_CLOSE_NS

def to_dollars(nano: int) -> float:
    return nano / NANOS_PER_DOLLAR

print('Imports OK')
print('DBN_PATH:', DBN_PATH)

---
## 1. Hawkes Process — Python Implementation

In [ ]:
@dataclass
class HawkesProcess:
    """Mirrors C++ HawkesProcess exactly."""
    mu: float = 0.0
    alpha: float = 0.0
    beta: float = 1.0
    lambda_cur: float = field(init=False)
    t_prev_ns: int = field(init=False, default=0)

    def __post_init__(self):
        self.lambda_cur = self.mu

    def _dt_sec(self, t_ns: int) -> float:
        return (t_ns - self.t_prev_ns) * 1e-9

    def decay(self, t_ns: int) -> None:
        """λ(t) = μ + exp(-β·Δt)·(λ(t⁻) − μ)"""
        dt = self._dt_sec(t_ns)
        self.lambda_cur = self.mu + math.exp(-self.beta * dt) * (self.lambda_cur - self.mu)
        self.t_prev_ns = t_ns

    def fire(self, t_ns: int) -> None:
        """Decay then add α jump."""
        self.decay(t_ns)
        self.lambda_cur += self.alpha

    def branching_ratio(self) -> float:
        return self.alpha / self.beta

    def is_stationary(self) -> bool:
        return self.branching_ratio() < 1.0

    def reset(self, t_ns: int) -> None:
        self.lambda_cur = self.mu
        self.t_prev_ns = t_ns


@dataclass
class HawkesTriple:
    """Mirrors C++ HawkesTriple."""
    buy:    HawkesProcess = field(default_factory=HawkesProcess)
    sell:   HawkesProcess = field(default_factory=HawkesProcess)
    cancel: HawkesProcess = field(default_factory=HawkesProcess)

    def update(self, t_ns: int, event_type: int) -> None:
        """Decay all three first, then fire the matching one."""
        self.buy.decay(t_ns)
        self.sell.decay(t_ns)
        self.cancel.decay(t_ns)
        if event_type == 0:
            self.buy.lambda_cur += self.buy.alpha
        elif event_type == 1:
            self.sell.lambda_cur += self.sell.alpha
        elif event_type == 2:
            self.cancel.lambda_cur += self.cancel.alpha

    def flow_imbalance(self) -> float:
        total = self.buy.lambda_cur + self.sell.lambda_cur
        if total == 0:
            return 0.5
        return self.buy.lambda_cur / total

    def total_mo_intensity(self) -> float:
        return self.buy.lambda_cur + self.sell.lambda_cur

    def reset(self, t_ns: int) -> None:
        self.buy.reset(t_ns)
        self.sell.reset(t_ns)
        self.cancel.reset(t_ns)

print('HawkesProcess and HawkesTriple defined')

---
## 2. Hawkes Process — Unit Tests

In [ ]:
PASS = '✓'
FAIL = '✗'

def approx_eq(a: float, b: float, tol=1e-10) -> bool:
    return abs(a - b) < tol

def check(name: str, cond: bool, extra: str = '') -> None:
    status = PASS if cond else FAIL
    print(f'  {status}  {name}' + (f'  [{extra}]' if extra else ''))
    if not cond:
        raise AssertionError(f'FAILED: {name}')

print('=== HawkesProcess Unit Tests ===')

# ── Test 1: initial state ──────────────────────────────────────────────────
print('\n[1] Initial state')
hp = HawkesProcess(mu=0.5, alpha=1.0, beta=5.0)
check('lambda_cur initialised to mu', approx_eq(hp.lambda_cur, 0.5))
check('branching_ratio = alpha/beta', approx_eq(hp.branching_ratio(), 0.2))
check('is_stationary (ratio<1)', hp.is_stationary())

# ── Test 2: pure decay formula ─────────────────────────────────────────────
print('\n[2] Decay formula: λ(t) = μ + exp(-β·Δt)·(λ(t⁻)−μ)')
hp = HawkesProcess(mu=0.5, alpha=1.0, beta=5.0)
hp.lambda_cur = 2.0      # artificially elevated
hp.t_prev_ns  = 0

dt_sec = 0.1             # 100 ms
t1_ns  = int(dt_sec * 1e9)
expected = 0.5 + math.exp(-5.0 * 0.1) * (2.0 - 0.5)
hp.decay(t1_ns)
check(f'decay(100ms): expected {expected:.6f}', approx_eq(hp.lambda_cur, expected),
      f'got {hp.lambda_cur:.6f}')

# ── Test 3: decay to baseline ──────────────────────────────────────────────
print('\n[3] Long decay converges to mu')
hp = HawkesProcess(mu=0.5, alpha=1.0, beta=5.0)
hp.lambda_cur = 3.0
hp.t_prev_ns  = 0
hp.decay(int(100 * 1e9))   # 100 seconds later
check('lambda ≈ mu after 100s', abs(hp.lambda_cur - 0.5) < 1e-6,
      f'got {hp.lambda_cur:.10f}')

# ── Test 4: fire adds alpha after decay ───────────────────────────────────
print('\n[4] Fire = decay then +α')
hp = HawkesProcess(mu=0.5, alpha=1.0, beta=5.0)
hp.lambda_cur = 0.5      # at baseline
hp.t_prev_ns  = 0
hp.fire(0)               # zero dt → pure alpha jump
check('fire(Δt=0): lambda = mu + alpha', approx_eq(hp.lambda_cur, 0.5 + 1.0),
      f'got {hp.lambda_cur}')

# ── Test 5: sequential fires accumulate correctly ─────────────────────────
print('\n[5] Two fires in quick succession')
hp = HawkesProcess(mu=0.5, alpha=1.0, beta=5.0)
hp.t_prev_ns = 0
hp.fire(0)                       # t=0, Δt=0 → λ = 0.5 + 1.0 = 1.5
t1 = int(0.01 * 1e9)             # 10 ms
lam_after_decay = 0.5 + math.exp(-5.0 * 0.01) * (1.5 - 0.5)
expected_second = lam_after_decay + 1.0
hp.fire(t1)
check(f'second fire at 10ms: {expected_second:.6f}', approx_eq(hp.lambda_cur, expected_second),
      f'got {hp.lambda_cur:.6f}')

# ── Test 6: non-stationary process detected ───────────────────────────────
print('\n[6] Non-stationary detection')
hp_bad = HawkesProcess(mu=0.5, alpha=5.0, beta=2.0)   # ratio=2.5 ≥ 1
check('branching_ratio=2.5 detected', approx_eq(hp_bad.branching_ratio(), 2.5))
check('is_stationary returns False', not hp_bad.is_stationary())

# ── Test 7: reset restores baseline ──────────────────────────────────────
print('\n[7] Reset')
hp = HawkesProcess(mu=0.5, alpha=1.0, beta=5.0)
hp.lambda_cur = 99.0
hp.reset(int(5e9))
check('lambda reset to mu', approx_eq(hp.lambda_cur, 0.5))
check('t_prev_ns updated', hp.t_prev_ns == int(5e9))

print('\n=== HawkesTriple Unit Tests ===')

# ── Test 8: update decays all, fires one ──────────────────────────────────
print('\n[8] Triple update fires only matching process')
ht = HawkesTriple(
    buy    = HawkesProcess(mu=0.5, alpha=1.0, beta=5.0),
    sell   = HawkesProcess(mu=0.5, alpha=0.8, beta=5.0),
    cancel = HawkesProcess(mu=2.0, alpha=1.5, beta=4.0),
)
# Elevate all three above baseline
ht.buy.lambda_cur    = 2.0;  ht.buy.t_prev_ns    = 0
ht.sell.lambda_cur   = 1.8;  ht.sell.t_prev_ns   = 0
ht.cancel.lambda_cur = 4.0;  ht.cancel.t_prev_ns = 0

dt_ns = int(0.05 * 1e9)   # 50 ms
ht.update(dt_ns, 0)       # buy event

exp_buy    = 0.5 + math.exp(-5.0*0.05)*(2.0-0.5) + 1.0
exp_sell   = 0.5 + math.exp(-5.0*0.05)*(1.8-0.5)
exp_cancel = 2.0 + math.exp(-4.0*0.05)*(4.0-2.0)

check(f'buy fired (λ={exp_buy:.5f})',    approx_eq(ht.buy.lambda_cur,    exp_buy),    f'{ht.buy.lambda_cur:.5f}')
check(f'sell decayed only (λ={exp_sell:.5f})',  approx_eq(ht.sell.lambda_cur,   exp_sell),   f'{ht.sell.lambda_cur:.5f}')
check(f'cancel decayed only (λ={exp_cancel:.5f})', approx_eq(ht.cancel.lambda_cur, exp_cancel), f'{ht.cancel.lambda_cur:.5f}')

# ── Test 9: flow imbalance ─────────────────────────────────────────────────
print('\n[9] Flow imbalance')
ht2 = HawkesTriple(
    buy    = HawkesProcess(mu=0.6, alpha=1.0, beta=5.0),
    sell   = HawkesProcess(mu=0.4, alpha=1.0, beta=5.0),
    cancel = HawkesProcess(mu=2.0, alpha=1.5, beta=4.0),
)
# At baseline: buy=0.6, sell=0.4
imb = ht2.flow_imbalance()
check(f'imbalance = 0.6/(0.6+0.4) = 0.6', approx_eq(imb, 0.6), f'{imb}')

# ── Test 10: equal buy+sell gives imbalance=0.5 ───────────────────────────
ht3 = HawkesTriple(
    buy    = HawkesProcess(mu=1.0, alpha=1.0, beta=5.0),
    sell   = HawkesProcess(mu=1.0, alpha=1.0, beta=5.0),
    cancel = HawkesProcess(mu=2.0, alpha=1.5, beta=4.0),
)
check('equal mu → imbalance=0.5', approx_eq(ht3.flow_imbalance(), 0.5))

print('\nAll Hawkes tests passed.')

---
## 3. Hawkes Intensity Visualisation — Synthetic Events

In [ ]:
rng = np.random.default_rng(42)

def simulate_hawkes_thinning(mu, alpha, beta, T_sec=60.0, seed=42):
    """Ogata thinning algorithm for a univariate Hawkes process."""
    rng = np.random.default_rng(seed)
    events = []
    t = 0.0
    hp = HawkesProcess(mu=mu, alpha=alpha, beta=beta)
    hp.t_prev_ns = 0

    while t < T_sec:
        lam_bar = hp.lambda_cur + alpha   # upper bound (next fire would add alpha)
        dt = rng.exponential(1.0 / max(lam_bar, 1e-9))
        t += dt
        if t >= T_sec:
            break
        t_ns = int(t * 1e9)
        hp.decay(t_ns)
        lam_now = hp.lambda_cur
        if rng.random() < lam_now / lam_bar:
            hp.fire(t_ns)
            events.append(t)
    return events

def hawkes_intensity_curve(mu, alpha, beta, events_sec, t_grid):
    """Reconstruct λ(t) on t_grid given a list of event times."""
    lam = np.zeros(len(t_grid))
    for i, t in enumerate(t_grid):
        lam[i] = mu + sum(alpha * math.exp(-beta * (t - s)) for s in events_sec if s < t)
    return lam

# Parameters
params = dict(mu=0.5, alpha=1.2, beta=5.0)
events = simulate_hawkes_thinning(**params, T_sec=60.0)
t_grid = np.linspace(0, 60, 3000)
lam    = hawkes_intensity_curve(**params, events_sec=events, t_grid=t_grid)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax = axes[0]
ax.plot(t_grid, lam, color='steelblue', lw=1.2, label='λ(t)')
ax.axhline(params['mu'], color='tomato', lw=1, ls='--', label='μ (baseline)')
ax.axhline(params['mu'] / (1 - params['alpha']/params['beta']),
           color='goldenrod', lw=1, ls=':', label='E[λ] stationary mean')
ax.set_ylabel('Intensity λ(t)')
ax.set_title(f'Hawkes Intensity  |  μ={params["mu"]}  α={params["alpha"]}  β={params["beta"]}  '
             f'(branching={params["alpha"]/params["beta"]:.2f})')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.eventplot(events, orientation='horizontal', colors='steelblue', lineoffsets=0.5, linelengths=0.8)
ax2.set_yticks([])
ax2.set_ylabel('Events')
ax2.set_xlabel('Time (seconds)')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('hawkes_intensity_synthetic.png', bbox_inches='tight')
plt.show()
print(f'Simulated {len(events)} events in 60 seconds')

In [ ]:
# Compare stationary vs non-stationary
configs = [
    ('Stationary  (α/β=0.5)',     dict(mu=0.5, alpha=1.0, beta=2.0)),
    ('Near-critical (α/β=0.9)',   dict(mu=0.2, alpha=1.8, beta=2.0)),
    ('Explosive (α/β=1.2)',       dict(mu=0.2, alpha=2.4, beta=2.0)),
]

fig, axes = plt.subplots(len(configs), 1, figsize=(12, 8), sharex=True)
for ax, (label, p) in zip(axes, configs):
    evts = simulate_hawkes_thinning(**p, T_sec=30.0, seed=1)
    lam  = hawkes_intensity_curve(**p, events_sec=evts, t_grid=np.linspace(0,30,1500))
    ax.plot(np.linspace(0,30,1500), lam, lw=1.2)
    ax.axhline(p['mu'], color='tomato', lw=0.8, ls='--')
    ax.set_ylabel('λ(t)')
    ax.set_title(f'{label}  ({len(evts)} events)')
    ax.grid(alpha=0.3)
axes[-1].set_xlabel('Time (seconds)')
plt.suptitle('Hawkes Intensity — Stationarity Regimes', y=1.01)
plt.tight_layout()
plt.savefig('hawkes_stationarity_regimes.png', bbox_inches='tight')
plt.show()

---
## 4. Order Book — Python Implementation

In [ ]:
from enum import Enum

class EventType(Enum):
    Add    = 'A'
    Cancel = 'C'
    Modify = 'M'
    Trade  = 'T'
    Fill   = 'F'
    Clear  = 'R'

@dataclass
class OrderEvent:
    timestamp_ns: int
    type: EventType
    order_id: int
    size: int
    price: int    # nanodollars
    side: int     # 1=bid, -1=ask

@dataclass
class Level:
    price: int = 0
    total_quantity: int = 0
    shares_ahead: int = 0
    we_have_order: bool = False
    orders: OrderedDict = field(default_factory=OrderedDict)  # order_id → size (FIFO)

    def add_order(self, order_id: int, size: int) -> None:
        self.orders[order_id] = size
        self.total_quantity += size

    def cancel_order(self, order_id: int, cancelled_size: int) -> bool:
        """Returns True if fully cancelled."""
        if order_id not in self.orders:
            return False
        cur = self.orders[order_id]
        if cancelled_size >= cur:
            self.total_quantity -= cur
            del self.orders[order_id]
            return True
        else:
            self.orders[order_id] -= cancelled_size
            self.total_quantity  -= cancelled_size
            return False

    def on_trade(self, executed_size: int) -> None:
        """Reduce total_qty and shares_ahead by executed amount."""
        self.total_quantity -= executed_size
        self.shares_ahead    = max(0, self.shares_ahead - executed_size)

    def empty(self) -> bool:
        return self.total_quantity <= 0


@dataclass
class OrderMeta:
    price: int
    side: int


class OrderBook:
    def __init__(self):
        self.bids: dict[int, Level] = {}   # price → Level, best bid = max key
        self.asks: dict[int, Level] = {}   # price → Level, best ask = min key
        self.order_index: dict[int, OrderMeta] = {}

    def best_bid(self) -> int:
        return max(self.bids) if self.bids else 0

    def best_ask(self) -> int:
        return min(self.asks) if self.asks else 0

    def bid_depth(self) -> int:
        bb = self.best_bid()
        return self.bids[bb].total_quantity if bb else 0

    def ask_depth(self) -> int:
        ba = self.best_ask()
        return self.asks[ba].total_quantity if ba else 0

    def mid_price(self) -> int:
        return (self.best_bid() + self.best_ask()) // 2

    def spread(self) -> int:
        return self.best_ask() - self.best_bid()

    def is_uncrossed(self) -> bool:
        return self.spread() > 0

    def _get_or_create(self, price: int, side: int) -> Level:
        book = self.bids if side == 1 else self.asks
        if price not in book:
            book[price] = Level(price=price)
        return book[price]

    def _find_level(self, price: int, side: int) -> Optional[Level]:
        book = self.bids if side == 1 else self.asks
        return book.get(price)

    def _remove_level(self, price: int, side: int) -> None:
        book = self.bids if side == 1 else self.asks
        if price in book and book[price].empty():
            del book[price]

    def apply(self, ev: OrderEvent) -> None:
        if ev.type == EventType.Add:
            self._apply_add(ev)
        elif ev.type == EventType.Cancel:
            self._apply_cancel(ev)
        elif ev.type == EventType.Modify:
            self._apply_modify(ev)
        elif ev.type in (EventType.Trade, EventType.Fill):
            self._apply_trade(ev)
        elif ev.type == EventType.Clear:
            self.bids.clear(); self.asks.clear(); self.order_index.clear()

    def _apply_add(self, ev: OrderEvent) -> None:
        lv = self._get_or_create(ev.price, ev.side)
        lv.add_order(ev.order_id, ev.size)
        self.order_index[ev.order_id] = OrderMeta(ev.price, ev.side)

    def _apply_cancel(self, ev: OrderEvent) -> None:
        meta = self.order_index.get(ev.order_id)
        if not meta:
            return
        lv = self._find_level(meta.price, meta.side)
        if not lv:
            return
        fully_cancelled = lv.cancel_order(ev.order_id, ev.size)
        if fully_cancelled:
            del self.order_index[ev.order_id]
        if lv.empty():
            self._remove_level(meta.price, meta.side)

    def _apply_modify(self, ev: OrderEvent) -> None:
        meta = self.order_index.get(ev.order_id)
        if not meta:
            return
        lv = self._find_level(meta.price, meta.side)
        if not lv or ev.order_id not in lv.orders:
            return
        old_size = lv.orders[ev.order_id]
        lv.total_quantity += (ev.size - old_size)
        lv.orders[ev.order_id] = ev.size

    def _apply_trade(self, ev: OrderEvent) -> None:
        meta = self.order_index.get(ev.order_id)
        if not meta:
            return
        lv = self._find_level(meta.price, meta.side)
        if not lv:
            return
        lv.on_trade(ev.size)
        if lv.empty():
            self._remove_level(meta.price, meta.side)
            if ev.order_id in self.order_index:
                del self.order_index[ev.order_id]

print('OrderBook and Level defined')

---
## 5. Order Book — Unit Tests

In [ ]:
def make_ev(t, etype, oid, size, price, side):
    return OrderEvent(t, etype, oid, size, price, side)

# Convenience prices in nanodollars
BID1 = 420_000_000_000   # $420.00
BID2 = 419_900_000_000   # $419.90
ASK1 = 420_100_000_000   # $420.10
ASK2 = 420_200_000_000   # $420.20

print('=== OrderBook Unit Tests ===')

# ── Test 1: Add bid and ask orders ────────────────────────────────────────
print('\n[1] Add bid and ask orders')
book = OrderBook()
book.apply(make_ev(1, EventType.Add, 1001, 100, BID1, 1))
book.apply(make_ev(2, EventType.Add, 1002,  50, BID1, 1))   # second at same level
book.apply(make_ev(3, EventType.Add, 2001, 200, ASK1, -1))

check('best_bid = BID1',     book.best_bid() == BID1)
check('best_ask = ASK1',     book.best_ask() == ASK1)
check('bid_depth = 150',     book.bid_depth() == 150)
check('ask_depth = 200',     book.ask_depth() == 200)
check('mid = (420.00+420.10)/2', book.mid_price() == (BID1 + ASK1) // 2)
check('spread = $0.10',      book.spread() == ASK1 - BID1)
check('is_uncrossed',        book.is_uncrossed())

# ── Test 2: Full cancel removes level ────────────────────────────────────
print('\n[2] Full cancel clears level')
book2 = OrderBook()
book2.apply(make_ev(1, EventType.Add,    3001, 100, ASK2, -1))
book2.apply(make_ev(2, EventType.Cancel, 3001, 100, ASK2, -1))  # full cancel
check('ask level removed after full cancel', ASK2 not in book2.asks)
check('order_index cleared',               3001 not in book2.order_index)

# ── Test 3: Partial cancel reduces size ──────────────────────────────────
print('\n[3] Partial cancel reduces size')
book3 = OrderBook()
book3.apply(make_ev(1, EventType.Add,    4001, 300, BID2, 1))
book3.apply(make_ev(2, EventType.Cancel, 4001, 100, BID2, 1))   # partial
lv = book3.bids[BID2]
check('level still exists',       not lv.empty())
check('remaining size = 200',     lv.total_quantity == 200)
check('order still in index',     4001 in book3.order_index)
check('order size updated to 200', lv.orders[4001] == 200)

# ── Test 4: Modify changes size without priority reset ────────────────────
print('\n[4] Modify changes size in-place')
book4 = OrderBook()
book4.apply(make_ev(1, EventType.Add,    5001, 100, BID1, 1))
book4.apply(make_ev(2, EventType.Add,    5002, 200, BID1, 1))
book4.apply(make_ev(3, EventType.Modify, 5001,  60, BID1, 1))   # reduce 5001
lv = book4.bids[BID1]
check('total_qty = 60+200 = 260', lv.total_quantity == 260)
check('5001 size = 60',           lv.orders[5001] == 60)
check('5002 size unchanged = 200', lv.orders[5002] == 200)
check('FIFO: 5001 still before 5002', list(lv.orders.keys()) == [5001, 5002])

# ── Test 5: Trade reduces shares_ahead and total_qty ─────────────────────
print('\n[5] Trade reduces shares_ahead')
book5 = OrderBook()
book5.apply(make_ev(1, EventType.Add, 6001, 100, ASK1, -1))  # first in queue
book5.apply(make_ev(2, EventType.Add, 6002, 200, ASK1, -1))  # our order (for test)

lv = book5.asks[ASK1]
lv.shares_ahead = 100   # we have 6001 ahead of us
lv.we_have_order = True

# Fill 6001 fully
book5.apply(make_ev(3, EventType.Fill, 6001, 100, ASK1, -1))
lv = book5.asks[ASK1]   # level should still exist
check('shares_ahead → 0 after 6001 fills', lv.shares_ahead == 0)
check('total_qty = 200 remaining',         lv.total_quantity == 200)

# ── Test 6: Multiple levels, FIFO ordering preserved ─────────────────────
print('\n[6] Multi-level book, FIFO order')
book6 = OrderBook()
for oid, price, side, size in [
    (10, BID1, 1,  500), (11, BID2, 1,  300),
    (20, ASK1, -1, 400), (21, ASK2, -1, 200),
]:
    book6.apply(make_ev(1, EventType.Add, oid, size, price, side))

check('2 bid levels',  len(book6.bids) == 2)
check('2 ask levels',  len(book6.asks) == 2)
check('bid sorted best-first', list(sorted(book6.bids, reverse=True))[0] == BID1)
check('ask sorted best-first', list(sorted(book6.asks))[0] == ASK1)

# ── Test 7: Clear wipes book ──────────────────────────────────────────────
print('\n[7] Clear wipes entire book')
book7 = OrderBook()
book7.apply(make_ev(1, EventType.Add, 99, 100, BID1, 1))
book7.apply(make_ev(2, EventType.Clear, 0, 0, 0, 0))
check('bids empty after clear', len(book7.bids) == 0)
check('asks empty after clear', len(book7.asks) == 0)
check('order_index empty',      len(book7.order_index) == 0)

# ── Test 8: Cancel unknown order is a no-op ────────────────────────────────
print('\n[8] Cancel unknown order — no-op')
book8 = OrderBook()
book8.apply(make_ev(1, EventType.Add, 100, 100, BID1, 1))
pre_qty = book8.bid_depth()
book8.apply(make_ev(2, EventType.Cancel, 9999, 50, BID1, 1))  # unknown id
check('depth unchanged after cancel of unknown id', book8.bid_depth() == pre_qty)

print('\nAll OrderBook tests passed.')

---
## 6. Kinetic Queue — Fill Probability Model

In [ ]:
@dataclass
class KineticQueue:
    """
    Fill probability model based on power-law queue dynamics.

    P(fill | q, Q) = 1 - (q/Q)^gamma

    q     = shares ahead of our order
    Q     = total shares at the price level
    gamma = power law exponent (0.5–1.5; 1.0 = linear)
    """
    gamma: float = 1.0

    def fill_prob(self, q: int, Q: int) -> float:
        if Q <= 0 or q < 0:
            return 0.0
        if q == 0:
            return 1.0
        if q >= Q:
            return 0.0
        return 1.0 - (q / Q) ** self.gamma

    def expected_queue_position(self, Q: int, n_steps: int = 1000) -> float:
        """E[time to fill] proxy: mean position at which fill occurs."""
        fracs = np.linspace(0, 1, n_steps + 1)[:-1]   # q/Q from 0 to ~1
        probs = 1 - fracs ** self.gamma
        return float(np.trapz(fracs * probs, fracs) / np.trapz(probs, fracs))

print('KineticQueue defined')

---
## 7. Kinetic Queue — Unit Tests

In [ ]:
print('=== KineticQueue Unit Tests ===')

kq = KineticQueue(gamma=1.0)

# ── Test 1: boundary conditions ────────────────────────────────────────────
print('\n[1] Boundary conditions')
check('q=0 → P=1.0 (front of queue)', approx_eq(kq.fill_prob(0, 500), 1.0))
check('q=Q → P=0.0 (last in queue)',  approx_eq(kq.fill_prob(500, 500), 0.0))
check('q>Q → P=0.0 (impossible)',     approx_eq(kq.fill_prob(600, 500), 0.0))
check('Q=0 → P=0.0 (empty level)',    approx_eq(kq.fill_prob(0, 0), 0.0))

# ── Test 2: linear gamma=1 ─────────────────────────────────────────────────
print('\n[2] Linear gamma=1: P = 1 − q/Q')
for q, Q, expected in [(0,100,1.0),(50,100,0.5),(25,100,0.75),(99,100,0.01)]:
    check(f'q={q}/Q={Q} → P={expected}', approx_eq(kq.fill_prob(q, Q), expected))

# ── Test 3: gamma<1 → (q/Q)^γ > q/Q → P lower than linear ───────────────
# For 0 < x < 1: x^γ > x when γ < 1, so 1 − x^γ < 1 − x = P_linear
print('\n[3] γ=0.5: (q/Q)^0.5 > q/Q for mid-queue → P lower than linear')
kq_concave = KineticQueue(gamma=0.5)
p_linear = kq.fill_prob(50, 100)
p_concave = kq_concave.fill_prob(50, 100)
check(f'P(γ=0.5)={p_concave:.3f} < P_linear={p_linear:.3f}', p_concave < p_linear)

# ── Test 4: gamma>1 → (q/Q)^γ < q/Q → P higher than linear ──────────────
# For 0 < x < 1: x^γ < x when γ > 1, so 1 − x^γ > 1 − x = P_linear
print('\n[4] γ=1.5: (q/Q)^1.5 < q/Q for mid-queue → P higher than linear')
kq_convex = KineticQueue(gamma=1.5)
p_convex = kq_convex.fill_prob(50, 100)
check(f'P(γ=1.5)={p_convex:.3f} > P_linear={p_linear:.3f}', p_convex > p_linear)

# ── Test 5: monotone decreasing in q ──────────────────────────────────────
print('\n[5] P(q, Q) is monotone decreasing in q')
Q = 1000
for gamma in [0.5, 1.0, 1.5]:
    kq_test = KineticQueue(gamma=gamma)
    probs = [kq_test.fill_prob(q, Q) for q in range(0, Q+1, 50)]
    monotone = all(probs[i] >= probs[i+1] for i in range(len(probs)-1))
    check(f'γ={gamma}: P monotone decreasing in q', monotone)

# ── Test 6: explicit values for common gammas ──────────────────────────────
print('\n[6] Explicit values: q=250, Q=1000 (25% queue position)')
expected_vals = {0.5: 1-(0.25**0.5), 1.0: 0.75, 1.5: 1-(0.25**1.5)}
for gamma, exp_val in expected_vals.items():
    p = KineticQueue(gamma=gamma).fill_prob(250, 1000)
    check(f'γ={gamma}: P={p:.5f} ≈ {exp_val:.5f}', approx_eq(p, exp_val))

print('\nAll KineticQueue tests passed.')

---
## 8. Kinetic Queue — Visualisation

In [ ]:
q_frac = np.linspace(0, 1, 500)   # q/Q
gammas = [0.5, 0.75, 1.0, 1.25, 1.5]
colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(gammas)))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: fill probability vs queue position
ax = axes[0]
for gamma, color in zip(gammas, colors):
    p = 1 - q_frac ** gamma
    ax.plot(q_frac * 100, p * 100, label=f'γ={gamma}', color=color, lw=2)
ax.axvline(50, color='gray', lw=0.8, ls='--', label='50% queue depth')
ax.set_xlabel('Shares Ahead / Total Queue  (%)')
ax.set_ylabel('Fill Probability  (%)')
ax.set_title('Kinetic Queue Fill Probability\nP(fill | q, Q) = 1 − (q/Q)^γ')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_xlim(0, 100); ax.set_ylim(0, 100)

# Right: marginal value of queue priority (dP/dq)
ax2 = axes[1]
for gamma, color in zip(gammas, colors):
    q_frac_inner = q_frac[1:]   # avoid div-by-zero at 0
    dPdq = -gamma * q_frac_inner ** (gamma - 1)  # derivative wrt q/Q
    ax2.plot(q_frac_inner * 100, -dPdq, label=f'γ={gamma}', color=color, lw=2)
ax2.set_xlabel('Shares Ahead / Total Queue  (%)')
ax2.set_ylabel('|dP/d(q/Q)| — Marginal Value of Priority')
ax2.set_title('Marginal Fill-Probability Gain\nfrom Moving One Position Forward')
ax2.legend(fontsize=9)
ax2.set_xlim(0, 100)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('kinetic_queue_fill_prob.png', bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: fill probability as a function of q and Q (absolute shares)
Q_vals = np.arange(100, 1100, 100)
q_vals = np.arange(0,   1100, 100)
gamma  = 1.0
kq_ref = KineticQueue(gamma=gamma)

grid = np.zeros((len(q_vals), len(Q_vals)))
for i, q in enumerate(q_vals):
    for j, Q in enumerate(Q_vals):
        grid[i, j] = kq_ref.fill_prob(int(q), int(Q))

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(grid, origin='lower', aspect='auto', cmap='RdYlGn',
               extent=[Q_vals[0], Q_vals[-1], q_vals[0], q_vals[-1]],
               vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Fill Probability')
ax.set_xlabel('Total Queue Size Q (shares)')
ax.set_ylabel('Shares Ahead q')
ax.set_title(f'Fill Probability Heatmap  (γ={gamma})')
# Diagonal q=Q → P=0
diag = np.linspace(100, 1000, 100)
ax.plot(diag, diag, 'k--', lw=1, alpha=0.5, label='q=Q boundary')
ax.legend()
plt.tight_layout()
plt.savefig('kinetic_queue_heatmap.png', bbox_inches='tight')
plt.show()

---
## 9. Databento MBO Data — Inspection

In [ ]:
import databento as db
from datetime import datetime, timezone

# Load from parquet cache (pre-filtered to session hours, ~2.5s vs ~110s for to_df())
CACHE_PATH = str(NOTEBOOK_DIR / 'data' / 'MSFT_20240303_20240314_mbo_cache.parquet')
df_raw = pd.read_parquet(CACHE_PATH)
df_raw.index = pd.to_datetime(df_raw['ts_ns'], utc=True)
df_raw = df_raw.drop(columns=['ts_ns'])

print(f'Loaded {len(df_raw):,} session records')
print(f'Columns: {list(df_raw.columns)}')
print(f'Date range: {df_raw.index[0]}  →  {df_raw.index[-1]}')

In [ ]:
# Display first 20 MBO records with human-readable fields
# Work on a small head slice — avoid apply() on all 17M rows
SAMPLE_N = 5000
df = df_raw.head(SAMPLE_N).copy()

if 'price' in df.columns:
    df['price_$'] = df['price'] / NANOS_PER_DOLLAR

def classify_hawkes(row):
    action = row.get('action', '')
    side   = row.get('side', '')
    if action == 'F':
        return 'buy_mo' if side == 'A' else 'sell_mo'
    if action == 'C':
        return 'cancel'
    return 'other'

df['hawkes_type'] = df.apply(classify_hawkes, axis=1)

display_cols = [c for c in ['action','side','price_$','size','order_id','hawkes_type']
                if c in df.columns]

print(f'=== First 20 MBO Records (of {len(df_raw):,} total) ===')
with pd.option_context('display.max_columns', None, 'display.width', 160,
                        'display.float_format', '{:.5f}'.format):
    print(df[display_cols].head(20).to_string())

In [ ]:
# Event type distribution — vectorized on full df_raw
print('=== Event Type Counts (full dataset) ===')
if 'action' in df_raw.columns:
    action_counts = df_raw['action'].value_counts()
    action_map    = {'A':'Add','C':'Cancel','M':'Modify','T':'Trade','F':'Fill','R':'Clear'}
    for code, cnt in action_counts.items():
        print(f'  {code} ({action_map.get(code, "?"):8s})  {cnt:>10,}  ({cnt/len(df_raw)*100:.1f}%)')
print(f'\n  TOTAL            {len(df_raw):>10,}')

# Hawkes classification — on the small sample only
print('\n=== Hawkes Classification (first 5000 records) ===')
h_counts = df['hawkes_type'].value_counts()
for label, cnt in h_counts.items():
    print(f'  {label:15s}  {cnt:>8,}  ({cnt/len(df)*100:.1f}%)')

In [ ]:
# All records in df_raw are already session-filtered (13:30–21:00 UTC)
df_session = df_raw
print(f'Session events: {len(df_session):,}')

fills = df_session[df_session['action'] == 'F']
print(f'Fill records:   {len(fills):,}')

if len(fills) > 0:
    f_disp = fills.head(15).copy()
    f_disp['price_$'] = f_disp['price'] / NANOS_PER_DOLLAR
    disp_cols = [c for c in ['action','side','price_$','size'] if c in f_disp.columns]
    print('\n=== Sample Fill Records (Hawkes events) ===')
    with pd.option_context('display.max_columns', None, 'display.width', 160,
                            'display.float_format', '{:.5f}'.format):
        print(f_disp[disp_cols].to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Event volume over time (5-minute bins)
ax = axes[0]
ts_seconds = df_session.index.astype('int64') * 1e-9
ts_minutes = (ts_seconds % 86400) / 60
bins = np.arange(ts_minutes.min(), ts_minutes.max(), 5)
ax.hist(ts_minutes, bins=bins, color='steelblue', alpha=0.8, edgecolor='none')
ax.set_xlabel('Minutes since midnight UTC')
ax.set_ylabel('Events per 5-min bin')
ax.set_title('MSFT MBO Event Volume by Time of Day')
ax.axvline(13.5*60, color='green', lw=1.5, ls='--', label='09:30 ET open')
ax.axvline(21.0*60, color='red',   lw=1.5, ls='--', label='16:00 ET close')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Price distribution
ax2 = axes[1]
if 'price' in df_session.columns:
    prices = df_session['price'] / NANOS_PER_DOLLAR
    valid_prices = prices[(prices > 300) & (prices < 600)]
    ax2.hist(valid_prices, bins=80, color='tomato', alpha=0.8, edgecolor='none')
ax2.set_xlabel('Price ($)')
ax2.set_ylabel('Order Count')
ax2.set_title('MSFT MBO Price Distribution (session)')
ax2.grid(alpha=0.3)

plt.suptitle('MSFT MBO Data — 2024-03-03 to 2024-03-14', y=1.02)
plt.tight_layout()
plt.savefig('databento_data_overview.png', bbox_inches='tight')
plt.show()

---
## 10. Integration — Hawkes Intensity on Real Market Data

In [ ]:
# Replay one day of real MSFT data through the Hawkes model.
params_buy    = dict(mu=0.5, alpha=1.2, beta=5.0)
params_sell   = dict(mu=0.5, alpha=1.2, beta=5.0)
params_cancel = dict(mu=2.0, alpha=1.5, beta=4.0)

ht_real = HawkesTriple(
    buy    = HawkesProcess(**params_buy),
    sell   = HawkesProcess(**params_sell),
    cancel = HawkesProcess(**params_cancel),
)

# Filter to 2024-03-04 (df_raw already session-filtered)
day_mask = (df_raw.index >= pd.Timestamp('2024-03-04', tz='UTC')) & \
           (df_raw.index <  pd.Timestamp('2024-03-05', tz='UTC'))
df_day = df_raw[day_mask]
print(f'Records for 2024-03-04 session: {len(df_day):,}')

# Vectorized Hawkes event type: 0=buy_mo, 1=sell_mo, 2=cancel, -1=other
ht_arr = np.full(len(df_day), -1, dtype=np.int8)
fill_ask = (df_day['action'] == 'F') & (df_day['side'] == 'A')
fill_bid = (df_day['action'] == 'F') & (df_day['side'] == 'B')
cancel   = (df_day['action'] == 'C')
ht_arr[fill_ask.values] = 0
ht_arr[fill_bid.values] = 1
ht_arr[cancel.values]   = 2

ts_arr_day = df_day.index.astype('int64').values

records_lam_buy    = []
records_lam_sell   = []
records_lam_cancel = []
records_imbalance  = []
records_ts_out     = []

SAMPLE_EVERY = 2000
ht_real.reset(int(ts_arr_day[0]))

for i, (ts_ns, ht) in enumerate(zip(ts_arr_day, ht_arr)):
    ht_real.update(int(ts_ns), int(ht))
    if i % SAMPLE_EVERY == 0:
        records_ts_out.append(int(ts_ns))
        records_lam_buy.append(ht_real.buy.lambda_cur)
        records_lam_sell.append(ht_real.sell.lambda_cur)
        records_lam_cancel.append(ht_real.cancel.lambda_cur)
        records_imbalance.append(ht_real.flow_imbalance())

print(f'Events processed: {len(df_day):,}')
print(f'Snapshots:        {len(records_ts_out):,}')

In [ ]:
if records_ts_out:
    ts_arr  = np.array(records_ts_out)
    ts_mins = (ts_arr % int(86400e9)) / 1e9 / 60   # minutes since midnight UTC

    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

    ax = axes[0]
    ax.plot(ts_mins, records_lam_buy,  color='green',  lw=0.8, label='λ_buy',  alpha=0.9)
    ax.plot(ts_mins, records_lam_sell, color='tomato', lw=0.8, label='λ_sell', alpha=0.9)
    ax.axhline(params_buy['mu'],  color='green',  lw=0.6, ls=':', alpha=0.5)
    ax.axhline(params_sell['mu'], color='tomato', lw=0.6, ls=':', alpha=0.5)
    ax.set_ylabel('λ (events/sec)')
    ax.set_title('Hawkes Intensities — MSFT 2024-03-04 (09:30–16:00 ET)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

    ax2 = axes[1]
    ax2.plot(ts_mins, records_lam_cancel, color='purple', lw=0.8, label='λ_cancel', alpha=0.9)
    ax2.axhline(params_cancel['mu'], color='purple', lw=0.6, ls=':', alpha=0.5)
    ax2.set_ylabel('λ_cancel (events/sec)')
    ax2.legend(fontsize=9)
    ax2.grid(alpha=0.3)

    ax3 = axes[2]
    ax3.plot(ts_mins, records_imbalance, color='steelblue', lw=0.8, alpha=0.9)
    ax3.axhline(0.5, color='gray',   lw=0.8, ls='--', label='neutral (0.5)')
    ax3.axhline(0.6, color='green',  lw=0.6, ls=':', alpha=0.5, label='buy pressure (>0.6)')
    ax3.axhline(0.4, color='tomato', lw=0.6, ls=':', alpha=0.5, label='sell pressure (<0.4)')
    ax3.set_ylabel('Flow Imbalance')
    ax3.set_xlabel('Minutes since midnight UTC (13:30=open, 21:00=close)')
    ax3.set_ylim(0, 1)
    ax3.legend(fontsize=9)
    ax3.grid(alpha=0.3)

    et_open  = 13.5 * 60
    et_close = 21.0 * 60
    for a in axes:
        a.axvline(et_open,  color='green', lw=1.0, ls='--', alpha=0.4)
        a.axvline(et_close, color='red',   lw=1.0, ls='--', alpha=0.4)

    plt.tight_layout()
    plt.savefig('hawkes_real_data.png', bbox_inches='tight')
    plt.show()
else:
    print('No data recorded — check date range or session filter')

In [ ]:
if records_lam_buy:
    print('=== Hawkes Summary Statistics (2024-03-04 session) ===')
    for name, arr in [('λ_buy', records_lam_buy), ('λ_sell', records_lam_sell),
                      ('λ_cancel', records_lam_cancel), ('imbalance', records_imbalance)]:
        a = np.array(arr)
        print(f'  {name:12s}  mean={a.mean():.4f}  std={a.std():.4f}  '
              f'min={a.min():.4f}  max={a.max():.4f}')

    print(f'\n  Branching ratios:')
    print(f'    buy:    α/β = {params_buy["alpha"]/params_buy["beta"]:.3f}')
    print(f'    sell:   α/β = {params_sell["alpha"]/params_sell["beta"]:.3f}')
    print(f'    cancel: α/β = {params_cancel["alpha"]/params_cancel["beta"]:.3f}')

---
## 11. Notes — Next Steps

| Component | Status | Next |
|---|---|---|
| `HawkesProcess` | ✓ Implemented & tested | MLE calibration on real data |
| `HawkesTriple` | ✓ Implemented & tested | Multi-variate (cross-excitation) upgrade |
| `OrderBook` | ✓ Implemented & tested | Attach to Databento replay, compare with C++ |
| `KineticQueue` | ✓ Model defined & tested | Estimate γ from fill data |
| Integration | ✓ Hawkes on real data | Feed full week, overlay bid/ask mid |

**Calibration:** Run MLE on Week 1 (2024-03-03 to 2024-03-08) to get `(μ, α, β)` per process.  
**Gamma estimation:** Use the fill records in Level.shares_ahead vs fill outcome to fit γ.